# RSNA Knee — Inference (submission notebook)

Offline, <=9h. Re-executes against the hidden test set at scoring time.
Attached datasets supply everything: `knee-code` (the package), `knee-weights`
(per-plane checkpoints). Output must be `/kaggle/working/submission.csv`.

Ensemble: each checkpoint declares its series type; each study fans out to every
model (strict typing, missing plane = model sits out) and rows merge by NaN-aware
mean. `USE_CONSTANT_PRIORS = True` reproduces the E000 mechanics-validation run.

In [ ]:
# No internet: code arrives as an attached dataset; the DICOM libs the Kaggle image
# lacks (pydicom, pylibjpeg + codecs) ship as wheels in that same dataset.
import sys
from pathlib import Path


def find_input(name: str) -> Path:
    # Kaggle's input layout moved (competitions now mount under a subdir); accept
    # every plausible dataset location and fail with a listing of actual mounts.
    base = Path("/kaggle/input")
    candidates = [base / name, base / "datasets" / "josiemachalek" / name, base / "datasets" / name]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    listing = {str(d): [x.name for x in d.iterdir()] for d in base.iterdir() if d.is_dir()}
    raise FileNotFoundError(f"{name} not found; mounts: {listing}")


CODE = find_input("knee-code")
WEIGHTS = find_input("knee-weights")
print("code:", CODE, "| weights:", WEIGHTS)

wheels = " ".join(str(w) for w in sorted((CODE / "wheels").glob("*.whl")))
assert wheels, f"no wheels under {CODE / 'wheels'}"
%pip install -q --no-deps {wheels}

sys.path.append(str(CODE))

from knee.infer import predict_studies
from knee.labels import SUBMISSION_COLUMNS

USE_CONSTANT_PRIORS = False  # True = E000 mechanics check, no model in the loop

In [ ]:
import pandas as pd

SLUG = "rsna-knee-abnormality-detection"
# Competition mounts under /kaggle/input/competitions/<slug> (verified on the train
# kernel 2026-08-31) or the legacy /kaggle/input/<slug>.
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP = next(p for p in candidates if (p / "test.csv").exists())
test = pd.read_csv(COMP / "test.csv")  # ~1300 studies at scoring time

In [ ]:
checkpoints = sorted(WEIGHTS.glob("*.pt"))
print([p.name for p in checkpoints])

In [ ]:
# The 9h budget lives here: each model decodes only its own series per study,
# and predict_studies never aborts on a bad study (a crash costs a submission).
if USE_CONSTANT_PRIORS:
    sub = pd.DataFrame(
        [[uid, *([0.5] * 12)] for uid in test["StudyInstanceUID"]],
        columns=list(SUBMISSION_COLUMNS),
    )
else:
    sub = predict_studies(COMP, checkpoints)

In [ ]:
assert list(sub.columns) == list(SUBMISSION_COLUMNS)
assert len(sub) == len(test) and not sub.isna().any().any()
sub.to_csv("submission.csv", index=False)  # exact filename required
sub.head()